# **Lab 5 - Context Engineering and RAG**
<p>COMP4146/7125 Prompt Engineering for Generative AI<br/>Semester 2, 2025/26, Dr. Shichao Ma, HKBU</p>


---


## Lab Overview

In this lab, you will implement **Retrieval-Augmented Generation (RAG)** end-to-end, based on Topic 5.

By the end of this lab, you should be able to:
1. Explain why unaided LLMs fail on private or fresh information (the frozen model problem).
2. Implement and compare **lexical retrieval** vs **neural retrieval (embeddings)**.
3. Build a basic **naive RAG** pipeline (index -> retrieve -> generate).
4. Tune key RAG controls such as chunking and `similarity_top_k`.
5. Apply prompt-level grounding constraints to improve faithfulness.

### Outline
1. Setup and local model check
2. Frozen model problem
3. Lexical retrieval baseline
4. Neural retrieval with LlamaIndex
5. Advanced RAG controls and exercises


## Setup: Install Required Packages
Install the required packages for this lab.


In [1]:
%pip install ollama llama-index llama-index-llms-ollama llama-index-embeddings-ollama


Note: you may need to restart the kernel to use updated packages.


## Setup: Pull Required Models
We use one generation model and one embedding model.

⚠️ Login your Ollama and make sure that **gemma3:4b-cloud** and **nomic-embed-text** these two models have been pulled.


## Setup: Imports and Helper Function


In [2]:
from pathlib import Path
import re
from collections import Counter
import ollama

LLM_MODEL = "gemma3:4b-cloud"
EMBED_MODEL = "nomic-embed-text"
DATA_DIR = Path("data")


def submit(messages, model=LLM_MODEL, temperature=0.0, num_ctx=4096):
    """Submit chat messages to Ollama and return text content."""
    response = ollama.chat(
        model=model,
        messages=messages,
        options={"temperature": temperature, "num_ctx": num_ctx},
    )
    return response["message"]["content"]


### Test Ollama Connection


In [3]:
test_response = submit([
    {"role": "user", "content": "Reply with exactly: Connection successful."}
])
print(test_response)


Connection successful.


---


## Part 1. The Frozen Model Problem

From Topic 5: LLMs are "frozen" at training time and do not know private local documents unless we retrieve and provide that context.


In [4]:
question = "In COMP4146, what is the exact late-submission penalty for homework?"

raw_answer = submit([
    {"role": "user", "content": question}
], temperature=0.0)

print("Question:", question)
print()
print("Raw LLM answer (no retrieval):")
print(raw_answer)


Question: In COMP4146, what is the exact late-submission penalty for homework?

Raw LLM answer (no retrieval):
Okay, let's break down the late submission penalty for COMP4146 (Data Structures and Algorithms) at the University of Melbourne.

As of the current information (October 26, 2023), here's the penalty structure:

*   **Up to 24 hours late:** -10% per 24 hours or part thereof.
*   **More than 24 hours late:** -20% per 24 hours or part thereof.

**Important Notes & Clarifications:**

*   **No submissions will be accepted after 72 hours late.** After this point, the assignment will receive a grade of 0.
*   **This applies to all homework assignments.** This includes programming assignments, quizzes, and any other assessed work.
*   **The penalty is applied *after* the original deadline.**  It's not a deduction from the total marks *before* the deadline.

**Where to Find the Official Policy:**

The most reliable source for this information is always the official course website and t

The model may guess, hallucinate, or stay uncertain because it has not seen your private course files.
This is exactly where RAG helps.


## Part 2. Retrieval Method 1: Lexical Retrieval (Keyword Matching)

This section corresponds to "Ctrl+F style" retrieval from Topic 5.


### 2.1 Create Local Context Documents
We create a small private knowledge base for this lab.


In [5]:
DATA_DIR.mkdir(exist_ok=True)

course_syllabus = """
COMP4146/7125 Prompt Engineering for Generative AI
Semester 2, 2025/26

Grading Breakdown:
- Lab assignments: 20%
- In-class quiz: 10%
- Group project: 20%
- Final project: 50%

The final project includes a group presentation and a personal report.
""".strip()

course_policies = """
Course Policies:
- Lab assignments are due every Monday 13:30, the commencement of the next lab session.
- Late submission without prior approval with valid reasons will not be accepted.
- Attendance is not recorded, but recommended for all lab sessions and lectures.
""".strip()

book_reviews = """
User review snippets about 'The Beach':
- I enjoy travel stories with moral ambiguity and flawed characters.
- I do not like slow starts, but I like high psychological tension.
- I prefer vivid setting descriptions over action-heavy writing.
""".strip()

(DATA_DIR / "Course_Syllabus.txt").write_text(course_syllabus, encoding="utf-8")
(DATA_DIR / "Course_Policies.txt").write_text(course_policies, encoding="utf-8")
(DATA_DIR / "Book_Reviews.txt").write_text(book_reviews, encoding="utf-8")

print("Created files:")
for p in sorted(DATA_DIR.glob("*.txt")):
    print("-", p)


Created files:
- data/Book_Reviews.txt
- data/Course_Policies.txt
- data/Course_Syllabus.txt


### 2.2 Snippetizing (Chunking)

Chunking rule of thumb from Topic 5:
- Under token limit
- One main idea per chunk
- Fits in prompt budget


In [6]:
def sliding_window_chunks(text, chunk_size=80, overlap=20):
    """Simple word-based chunking with overlap."""
    words = text.split()
    step = max(1, chunk_size - overlap)
    chunks = []

    for start in range(0, len(words), step):
        piece = words[start:start + chunk_size]
        if not piece:
            break
        chunks.append(" ".join(piece))
        if start + chunk_size >= len(words):
            break

    return chunks


snippets = []
for file_path in sorted(DATA_DIR.glob("*.txt")):
    text = file_path.read_text(encoding="utf-8")
    for i, chunk in enumerate(sliding_window_chunks(text, chunk_size=60, overlap=15)):
        snippets.append({
            "file": file_path.name,
            "chunk_id": i,
            "text": chunk,
        })

print(f"Total snippets: {len(snippets)}")
print()
print("Preview:")
for row in snippets[:3]:
    print(f"[{row['file']}#{row['chunk_id']}] {row['text'][:140]}...")


Total snippets: 3

Preview:
[Book_Reviews.txt#0] User review snippets about 'The Beach': - I enjoy travel stories with moral ambiguity and flawed characters. - I do not like slow starts, bu...
[Course_Policies.txt#0] Course Policies: - Lab assignments are due every Monday 13:30, the commencement of the next lab session. - Late submission without prior app...
[Course_Syllabus.txt#0] COMP4146/7125 Prompt Engineering for Generative AI Semester 2, 2025/26 Grading Breakdown: - Lab assignments: 20% - In-class quiz: 10% - Grou...


### 2.3 Build a Lexical Retriever


In [7]:
STOPWORDS = {
    "the", "a", "an", "is", "are", "of", "to", "for", "and", "or", "in", "on", "at", "with", "by", "what", "how"
}


def tokenize(text):
    return [w for w in re.findall(r"[A-Za-z0-9_]+", text.lower()) if w not in STOPWORDS]


def lexical_retrieve(query, corpus, top_k=3):
    q_terms = Counter(tokenize(query))
    scored = []

    for row in corpus:
        d_terms = Counter(tokenize(row["text"]))
        overlap = set(q_terms.keys()) & set(d_terms.keys())
        score = sum(min(q_terms[t], d_terms[t]) for t in overlap)
        if score > 0:
            scored.append((score, sorted(overlap), row))

    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]


### 2.4 Naive RAG Generation with Lexical Retrieval


In [8]:
def answer_with_lexical_rag(query, corpus, top_k=3):
    hits = lexical_retrieve(query, corpus, top_k=top_k)
    if not hits:
        return "No lexical context found.", []

    context_blocks = []
    for rank, (score, overlap, row) in enumerate(hits, start=1):
        block = f"[{rank}] ({row['file']}#{row['chunk_id']}) {row['text']}"
        context_blocks.append(block)

    prompt = """
You are a course assistant.
Answer only using the retrieved context.
If the answer is not in context, say: I don't know based on the provided context.

Retrieved Context:
{context}

Question: {question}
Answer:
""".strip().format(context="\n\n".join(context_blocks), question=query)

    answer = submit([
        {"role": "user", "content": prompt}
    ], temperature=0.0)

    return answer, hits


query = "What is the late-submission penalty and maximum late window?"
lex_answer, lex_hits = answer_with_lexical_rag(query, snippets, top_k=3)

print("Question:", query)
print()
print("Retrieved snippets:")
for score, overlap, row in lex_hits:
    print(f"- score={score}, overlap={overlap}, source={row['file']}#{row['chunk_id']}")

print()
print("Lexical RAG answer:")
print(lex_answer)


Question: What is the late-submission penalty and maximum late window?

Retrieved snippets:
- score=2, overlap=['late', 'submission'], source=Course_Policies.txt#0

Lexical RAG answer:
Late submission without prior approval with valid reasons will not be accepted.


### <b style="color:orange"> Task 1: Hallucination vs. Lexical RAG </b>

Try a query about course details (for example attendance or grading) and compare:
1. Raw LLM answer (no retrieval)
2. Lexical RAG answer

Write 2-3 lines on whether retrieval improved factual grounding.


In [9]:
# TODO: replace with your own test question
question = "What is the grading breakdown of COMP4146?"

# 1) Raw answer without retrieval
raw_answer = submit([
    {"role": "user", "content": question}
], temperature=0.0)

print("Raw answer:")
print(raw_answer)

Raw answer:
Okay, let's break down the grading breakdown for COMP4146 (Introduction to Artificial Intelligence) at the University of Waterloo. As of the Fall 2023 and Fall 2024 semesters, here's the detailed breakdown:

**Total Weight: 100%**

*   **Assignments (35%):**
    *   There will be 3-4 programming assignments throughout the term. These are designed to reinforce concepts learned in lectures and labs.  They are typically worth around 11-13% each.
*   **Midterm Exam (25%):**
    *   A written exam covering the material from the first half of the course.  It will likely include a mix of conceptual questions and problem-solving.
*   **Final Exam (25%):**
    *   A comprehensive written exam covering all the material from the entire course.  It will also likely include a mix of conceptual questions and problem-solving.
*   **Lab Component (15%):**
    *   This is a significant component and includes:
        *   **Lab Participation (5%):**  Active participation in the weekly labs i

In [10]:
# 2) Lexical RAG answer
lexical_answer, _ = answer_with_lexical_rag(question, snippets, top_k=3)

print("Lexical RAG answer:")
print(lexical_answer)

Lexical RAG answer:
The final project is 50% of the grade.


**Observation** (reference):
- The raw LLM answer may be generic or uncertain without private course context.
- Lexical RAG should recover grounded details from Course_Syllabus.txt

---


## Part 3. Retrieval Method 2: Neural Retrieval (Embeddings)

Now we move from exact keyword matching to semantic retrieval.


### 3.1 Configure LlamaIndex


In [13]:
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

# Compatibility across llama-index versions:
# Some versions accept request_timeout/model_name, others only accept model.
try:
    Settings.llm = Ollama(model=LLM_MODEL, request_timeout=360.0)
except TypeError:
    Settings.llm = Ollama(model=LLM_MODEL)

try:
    Settings.embed_model = OllamaEmbedding(model_name=EMBED_MODEL)
except TypeError:
    Settings.embed_model = OllamaEmbedding(model=EMBED_MODEL)

print("LlamaIndex configured successfully.")


LlamaIndex configured successfully.


### 3.2 Build a Vector Index (Naive RAG: Indexing Step)


In [14]:
documents = SimpleDirectoryReader(str(DATA_DIR)).load_data()
print(f"Loaded {len(documents)} document(s).")

index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(similarity_top_k=3)
print("Vector index built.")


2026-03-01 16:46:39,563 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-03-01 16:46:39,581 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"


Loaded 3 document(s).
Vector index built.


### 3.3 Query with Neural Retrieval and Inspect Sources


In [15]:
query = "What is the late-submission penalty and maximum late window?"
response = query_engine.query(query)

print("Neural RAG answer:")
print(response)

print()
print("Retrieved source nodes:")
source_nodes = getattr(response, "source_nodes", []) or []

for i, node in enumerate(source_nodes, start=1):
    metadata = getattr(node, "metadata", {}) or {}
    source_name = (
        metadata.get("file_name")
        or metadata.get("filename")
        or metadata.get("source")
        or "unknown"
    )

    score = getattr(node, "score", None)
    score_text = f"{score:.4f}" if isinstance(score, (int, float)) else "N/A"

    node_text = getattr(node, "text", None)
    if node_text is None and hasattr(node, "get_content"):
        node_text = node.get_content()
    node_text = str(node_text) if node_text is not None else ""

    print(f"[{i}] score={score_text}, source={source_name}")
    print(node_text[:180].replace("\n", " "), "...")
    print()


2026-03-01 16:46:42,546 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-03-01 16:46:43,898 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Neural RAG answer:
Late submissions without prior approval and valid reasons will not be accepted.

Retrieved source nodes:
[1] score=0.6183, source=Course_Policies.txt
Course Policies: - Lab assignments are due every Monday 13:30, the commencement of the next lab session. - Late submission without prior approval with valid reasons will not be acc ...

[2] score=0.5694, source=Course_Syllabus.txt
COMP4146/7125 Prompt Engineering for Generative AI Semester 2, 2025/26  Grading Breakdown: - Lab assignments: 20% - In-class quiz: 10% - Group project: 20% - Final project: 50%  Th ...

[3] score=0.5121, source=Book_Reviews.txt
User review snippets about 'The Beach': - I enjoy travel stories with moral ambiguity and flawed characters. - I do not like slow starts, but I like high psychological tension. - I ...



### 3.4 Lexical vs Neural Retrieval on a Rephrased Query

The query below avoids exact wording from the policy text.
Neural retrieval should usually be more robust than lexical retrieval.


In [16]:
rephrased_query = "What deduction applies if homework is turned in after the deadline, and how long is the grace period?"

lex_answer, _ = answer_with_lexical_rag(rephrased_query, snippets, top_k=3)
neural_answer = query_engine.query(rephrased_query)

print("Rephrased query:", rephrased_query)
print()
print("Lexical RAG answer:")
print(lex_answer)
print()
print("Neural RAG answer:")
print(neural_answer)


2026-03-01 16:46:45,701 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-03-01 16:46:46,206 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Rephrased query: What deduction applies if homework is turned in after the deadline, and how long is the grace period?

Lexical RAG answer:
No lexical context found.

Neural RAG answer:
Late submissions without prior approval and valid reasons will not be accepted.


### <b style="color:orange"> Task 2: Compare Lexical vs Embedding Retrieval </b>

Use one query about syllabus/policy details and compare:
1. `answer_with_lexical_rag(...)` (lexical retrieval)
2. `query_engine.query(...)` (embedding retrieval)

Write 2-3 lines on which method gives a better answer for your query and why.



In [17]:
# TODO: replace with your own query
compare_query = "What happens if students don't submit my assignment on time?"

# 1) Lexical retrieval answer
lexical_answer, lexical_hits = answer_with_lexical_rag(compare_query, snippets, top_k=3)

# 2) Embedding retrieval answer
embedding_answer = query_engine.query(compare_query)

print("Query:", compare_query)
print()
print("Lexical answer:")
print(lexical_answer)
print()
print("Embedding answer:")
print(embedding_answer)

print()
print("Top lexical hits:")
for score, overlap, row in lexical_hits:
    print(f"- score={score}, overlap={overlap}, source={row['file']}#{row['chunk_id']}")

print()
print("Top embedding source nodes:")
for i, node in enumerate(getattr(embedding_answer, "source_nodes", []) or [], start=1):
    md = getattr(node, "metadata", {}) or {}
    source_name = md.get("file_name") or md.get("filename") or md.get("source") or "unknown"
    print(f"- [{i}] source={source_name}")

# Observation (2-3 lines):
# -


2026-03-01 16:46:46,271 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-03-01 16:46:46,751 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Query: What happens if students don't submit my assignment on time?

Lexical answer:
No lexical context found.

Embedding answer:
Late submissions without prior approval and valid reasons will not be accepted.

Top lexical hits:

Top embedding source nodes:
- [1] source=Course_Policies.txt
- [2] source=Course_Syllabus.txt
- [3] source=Book_Reviews.txt


**Observation**: Neural(Embedding) Retrival provides more flexibility.

---


## Part 4. Advanced RAG Controls

Topic 5 highlights that strong RAG systems also optimize prompt usage and retrieval flow.


### 4.1 Prompt Control: Grounded Answers with Citations


In [18]:
from llama_index.core import PromptTemplate

qa_template = PromptTemplate(
    """
You are a strict teaching assistant.
Use only the context below to answer.
If the answer is not in the context, say: I don't know based on the provided context.
Include source filenames in square brackets, e.g., [Course_Policies.txt].

Context:
{context_str}

Question: {query_str}
Answer:
""".strip()
)

question = "How is COMP4146 graded?"

# Preferred path (supported in many versions)
try:
    grounded_engine = index.as_query_engine(similarity_top_k=3, text_qa_template=qa_template)
    response = grounded_engine.query(question)
    print(response)

# Fallback path for versions where text_qa_template is not accepted
except TypeError:
    print("Fallback: text_qa_template is unsupported in this llama-index version.")

    retriever = index.as_retriever(similarity_top_k=3)
    retrieved_nodes = retriever.retrieve(question)

    context_parts = []
    sources = []
    for node in retrieved_nodes:
        txt = getattr(node, "text", None)
        if txt is None and hasattr(node, "get_content"):
            txt = node.get_content()
        if txt:
            context_parts.append(str(txt))

        md = getattr(node, "metadata", {}) or {}
        src = md.get("file_name") or md.get("filename") or md.get("source")
        if src:
            sources.append(src)

    context_str = "\n\n".join(context_parts)
    prompt = qa_template.format(context_str=context_str, query_str=question)
    manual_response = submit([
        {"role": "user", "content": prompt}
    ], temperature=0.0)

    print(manual_response)
    if sources:
        print("Sources:", sorted(set(sources)))


2026-03-01 16:46:46,801 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-03-01 16:46:47,582 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


COMP4146 is graded as follows: Lab assignments (20%), In-class quiz (10%), Group project (20%), and Final project (50%). [Course_Syllabus.txt]


### 4.2 The Three Key Questions of RAG

For any RAG application, explicitly answer:
1. **What to retrieve?** (token, phrase, chunk, paragraph, entity, etc.)
2. **When to retrieve?** (once, every turn, every N tokens, adaptive)
3. **How to use retrieved info?** (pre-prompt context, re-ranking, filtering, citation constraints)


### <b style="color:orange"> Task 3: Design Your Own Mini-RAG Use Case </b>

Design one mini-RAG application and fill the template in the next cell.

Use this checklist:
1. `use_case_name`: one clear application title.
2. `what_to_retrieve`: what unit you retrieve (chunk/paragraph/table row/etc.) and what metadata you keep.
3. `when_to_retrieve`: when retrieval happens (every user query, every turn, or conditional).
4. `how_to_use_context`: how retrieved text is injected and what grounding rule you enforce.
5. `failure_mode`: one realistic failure case.
6. `mitigation`: one concrete fix.

Write each field in **one concise sentence**.


In [19]:
# Fill this template with your own design.
use_case_name = ""
what_to_retrieve = ""
when_to_retrieve = ""
how_to_use_context = ""
failure_mode = ""
mitigation = ""

print("Use case:", use_case_name)
print("What to retrieve:", what_to_retrieve)
print("When to retrieve:", when_to_retrieve)
print("How to use context:", how_to_use_context)
print("Failure mode:", failure_mode)
print("Mitigation:", mitigation)


Use case: 
What to retrieve: 
When to retrieve: 
How to use context: 
Failure mode: 
Mitigation: 


In [20]:
# Reference answer
use_case_name = "HKBU Student Handbook QA Assistant"
what_to_retrieve = "Paragraph-level chunks from handbook sections (attendance, assessment, misconduct) with metadata fields for semester and section title."
when_to_retrieve = "Retrieve at each user query and re-retrieve for follow-up turns when user intent shifts to a different policy topic."
how_to_use_context = "Inject top-3 retrieved chunks into the prompt, require citations [file/section], and force 'I don't know' when evidence is missing."
failure_mode = "Retriever returns an outdated policy chunk from a previous semester and the system gives incorrect advice."
mitigation = "Apply semester metadata filtering, then rerank by section relevance before generation to reduce stale-policy errors."

print("Use case:", use_case_name)
print("What to retrieve:", what_to_retrieve)
print("When to retrieve:", when_to_retrieve)
print("How to use context:", how_to_use_context)
print("Failure mode:", failure_mode)
print("Mitigation:", mitigation)


Use case: HKBU Student Handbook QA Assistant
What to retrieve: Paragraph-level chunks from handbook sections (attendance, assessment, misconduct) with metadata fields for semester and section title.
When to retrieve: Retrieve at each user query and re-retrieve for follow-up turns when user intent shifts to a different policy topic.
How to use context: Inject top-3 retrieved chunks into the prompt, require citations [file/section], and force 'I don't know' when evidence is missing.
Failure mode: Retriever returns an outdated policy chunk from a previous semester and the system gives incorrect advice.
Mitigation: Apply semester metadata filtering, then rerank by section relevance before generation to reduce stale-policy errors.


---


## Pitfalls and Extension

**Common pitfall:** retrieving too many irrelevant chunks can hurt answer quality.
- Fix: tighten chunking, reduce `top_k`, or add filtering/reranking.

**Optional extension:** implement a hybrid retriever that combines lexical score and embedding similarity.


## 🎯 Key Takeaways

1. RAG addresses the frozen model limitation by injecting external context at inference time.
2. Lexical retrieval is simple and transparent, but brittle to wording changes.
3. Neural retrieval with embeddings improves semantic matching for rephrased queries.
4. Chunking, `top_k`, and prompt constraints strongly affect factuality and quality.
5. LlamaIndex and LangChain can both implement RAG, with different abstractions and ergonomics.
6. Good RAG design starts with: what to retrieve, when to retrieve, and how to use it.



---


### <b style="color:orange"> Submission </b>
Make sure that you **1) successfully generate all outputs in this notebook and 2) finish all the tasks**.

Submit the updated notebook with the filename ***YourName_YourStudentID.ipynb*** to the Moodle.

